# 조건부 엣지로 경로 선택

검색 점수가 기준을 넘는지에 따라 답변 생성과 추가 자료 요청 중 어느 노드로 이동하게 할까요?

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** add_conditional_edges('inspect_score', choose)는 inspect_score가 끝난 직후 choose를 호출합니다. choose는 작업 결과를 만드는 노드가 아니라 다음 경로의 이름을 반환하는 함수입니다. 반환값과 실제 노드 이름이 다르면 path_map으로 대응 관계를 적고, 같으면 반환형 Literal이 가능한 경로를 문서화합니다.

**다른 상황에 적용.** 문서 검색 외에도 결제 금액이 승인 한도 이하면 approve, 초과하면 manual_review로 보내는 흐름에 같은 구조를 적용할 수 있습니다. 입력 필드와 임계값은 달라져도 조건 함수와 실제 작업 노드를 분리하는 원칙은 유지됩니다.

**비교와 경계.** 조건부 엣지는 경로만 고릅니다. 판단 결과를 상태에 남기면서 이동까지 한 번에 표현하려면 뒤에서 배우는 Command가 더 직접적입니다. 또한 0.7이라는 숫자는 LangGraph의 기본값이 아니라 이 실습에서 정한 정책이므로 평가 데이터 없이 일반 기준으로 사용할 수 없습니다.


## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 버전 확인

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 scikit-learn==1.9.0
from importlib.metadata import version
print('Python 패키지 준비 완료')
print('langgraph', version('langgraph'))

Note: you may need to restart the kernel to use updated packages.
Python 패키지 준비 완료
langgraph 1.2.11


## 상태와 분기 함수 정의

다음 코드에서 상태와 분기 함수 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `RouteState` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `inspect_score(state: RouteState)`, `choose(state: RouteState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다.


In [2]:
# 상태와 분기 함수 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class RouteState(TypedDict):
    rerank_score: float
    route: str
    answer: str

def inspect_score(state: RouteState) -> dict:
    return {'route': 'checked'}

def choose(state: RouteState) -> Literal['generate', 'request_more']:
    return 'generate' if state['rerank_score'] >= 0.7 else 'request_more'

## 두 목적지 노드 정의

다음 코드에서 두 목적지 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `generate(state: RouteState)`, `request_more(state: RouteState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `StateGraph(StateType)`은 상태 형식을 입력으로 받아 그래프 설계 객체를 만듭니다. `add_node(name, function)`은 그래프 이름과 실제 실행할 함수를 연결하며, 이 호출만으로 함수 본문이 실행되지는 않습니다.


In [3]:
# 두 목적지 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
def generate(state: RouteState) -> dict:
    return {'route': 'generate', 'answer': '근거로 답변 생성'}

def request_more(state: RouteState) -> dict:
    return {'route': 'request_more', 'answer': '추가 자료 요청'}

builder = StateGraph(RouteState)
builder.add_node('inspect_score', inspect_score)
builder.add_node('generate', generate)
builder.add_node('request_more', request_more)

## 조건부 엣지 연결

조건부 엣지 연결 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `add_edge(from, to)`는 두 노드의 고정 이동 순서를 등록합니다. `add_conditional_edges(source, path)`는 source가 끝난 뒤 path 함수를 호출해 다음 목적지를 고릅니다. `compile()`은 선언한 노드와 엣지로 `invoke()`할 실행 객체를 반환합니다. `draw_mermaid()`는 컴파일된 그래프의 노드와 연결을 Mermaid 문자열로 반환하며 그래프를 다시 실행하지 않습니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [4]:
# 조건부 엣지 연결 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
builder.add_edge(START, 'inspect_score')
builder.add_conditional_edges('inspect_score', choose)
builder.add_edge('generate', END)
builder.add_edge('request_more', END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	inspect_score(inspect_score)
	generate(generate)
	request_more(request_more)
	__end__([<p>__end__</p>]):::last
	__start__ --> inspect_score;
	inspect_score -.-> generate;
	inspect_score -.-> request_more;
	generate --> __end__;
	request_more --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 점수 하나를 바꿔 경로 비교

점수 하나를 바꿔 경로 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

**이 셀의 코드 읽기.** `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [5]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
base = {'route': '', 'answer': ''}
high = graph.invoke({**base, 'rerank_score': 0.82})
low = graph.invoke({**base, 'rerank_score': 0.41})
print('0.82:', high['route'], high['answer'])
print('0.41:', low['route'], low['answer'])
assert high['route'] == 'generate'
assert low['route'] == 'request_more'

0.82: generate 근거로 답변 생성
0.41: request_more 추가 자료 요청
